In [3]:
import os
import time
import json
import requests
import pandas as pd
import lyricsgenius
from openai import OpenAI
from typing import Optional, Tuple
from tqdm import tqdm
import numpy as np
import re
import base64
import ast

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

tqdm.pandas(desc="fetching lyrics")

In [4]:
df_genres = pd.read_csv('/Users/vasilii/Documents/python/study/masters/flask/dashboard_data/df_genres.csv')

In [7]:
df_genres.sample(5)

,year,platform,raw_artist,raw_song,matched_artist,matched_song,ai_suggestions,lyrics,genius_id,lrclib_id,genres_yandex,genres_musicbrainz,genres_itunes,genres_lastfm,chosen_genre,extracted_brands,mtld
2078,2019-09-08,youtube,Воровайки,Мотыльки,Воровайки,Мотыльки,[],"Вот по рельсам колёса стучат\nНа тайгу малолеток везут\nПацаны за решёткой не спят\nПро судьбу свою тихо поют\n\nПацаны за решёткой не спят\nПро судьбу свою тихо поют\nПолетят мотыльки над тайгой\nБудто сотни загубленных судеб\n\nИ ни лагерь, ни подлый конвой\nНет, никто, лишь Господь их осудит\n\nДождь по капелькам бьёт о стекло\n\nНо по капелькам можно понять\nЧто на волю глядят мотыльки\nНо её в темноте не видать\nЧто на волю глядят мотыльки\n\nНо её в темноте не видать\nПолетят мотыльки над тайгой\nБудто сотни загубленных судеб\nИ ни лагерь, ни подлый конвой\n\nНет, никто, лишь Господь их осудит\n\nГде-то там дискотеки звенят\nДа девчонки приходят в кино\n\nТолько жаль, позабудут ребят\nДля которых в решётку окно\nТолько жаль, позабудут ребят\nДля которых в решётку окно\n\nПолетят мотыльки над тайгой\nБудто сотни загубленных судеб\nИ ни лагерь, ни подлый конвой\nНет, никто, лишь Господь их осудит\n\nПолетят мотыльки над тайгой\nБудто сотни загубленных судеб\nИ ни лагерь, ни подлый конвой\nНет, никто, лишь Господь их осудит\n\nИ ни лагерь, ни подлый конвой\nНет, никто, лишь Господь их осудит",NaN,11431204.0,['shanson'],[],['Chanson in Russian'],"['chanson', 'glam chanson', 'I listen to this crap just for lulz', 'kircore', 'russian']",other,[],36.400000
671,2009-05-25,radio,Инфинити,Слёзы-вода,Инфинити,Слёзы-вода,[],"Слёзы-вода\n\nСлёзы-вода\n\nЯ останусь ветром\nШаг за шагом, за мечтой\nЯ останусь небом\nЛишь бы рядом быть всегда с тобой\n\nЯ в тебе искала\nИ нашла любовь свою\nТы любил так мало\nТы сказал: ""Ухожу""\n\nСлёзы-вода, между нами облака\nЯ схожу с ума, не могу я без тебя\nСлёзы-вода, между нами облака\nЯ люблю тебя, но в ответ лишь тишина\n\nСлёзы-вода\nСлёзы-вода\n\nБьётся моё сердце\nВновь сегодня, как вчера\nКрасная помада\nМини-юбка, туфли — как всегда\n\nПлачу я и таю\nТихо в листьях прошепчу\nВспоминаю, знаешь\nПросто я ухожу\n\nСлёзы-вода, между нами облака\nЯ схожу с ума, не могу я без тебя\nСлёзы-вода, между нами облака\nЯ люблю тебя, но в ответ лишь тишина\n\nСлёзы-вода\n\nСлёзы-вода\n",NaN,NaN,['ruspop'],[],['Pop'],"['pop', 'russian', 'electronic', 'dance', 'House']",pop,[],29.251660
2199,2024-04-14,youtube,Alblak 52,+7(952)812,Alblak 52,+7(952)812,[],"Это второй\nThat’s a Krishtall\nАу-у, YEEI, а\n52 (Алло)\nДа здравствует Санкт-Петербург (А), и это город наш (YEEI)\nЯ каждый свой новый куплет валю как никогда (YEEI, а)\nАльбом, он чисто мой, никому его не продам (Он мой)\nНе думаю о том (YEEI), как хорошо было вчера (А-а; мне пох)\nМеняю города (А)\nПредставляю район — у меня есть репертуар (YEEI, 2−3)\nНикогда не просил, но всегда где-то доставал (Где?)\nЧем больше денег (А), тем больше мне нравится Москва (А)\nНо в Питере душа (YEEI), в Питере семья (YEEI)\nВ Питере братва (А, а), там знают наши имена (52)\n+7(952)8−1-2 (Алло)\nЭто второй альбом (А), вторая глава (Второй)\nНе думал, не гадал, всё, что я делал, — рэповал (Всегда)\nАндеграунд — это не броуки в протёртых штанах (Пошёл на хуй)\nНужно прожить мою жизнь, чтоб так же, как я, слагать (Ага)\nНужно мой рэп услышать (YEEI), чтоб точно его понять\n52 (Алло)\nДа здравствует Санкт-Петербург (А), и это город наш (YEEI)\nЯ каждый свой новый куплет валю как никогда (YEEI, а)\nАльбом, он чисто мой, никому его не продам (Он мой)\nНе думаю о том (YEEI), как хорошо было вчера (Ага)\nДа здравствует 52\nДа здравствует Петербург, да здравствует 52\nДа здравствует Петербург, да здравствует 52 (Ау; YEEI, а)\nДа здравствует 52 (Ау), YEEI, long live (Это второй)",NaN,NaN,"['rusrap', 'rap']",[],['Hip-Hop/Rap'],"['rap', 'Hip-Hop', 'trap', 'russian']",rap,[],36.736079
1817,2020-08-07,youtube,Konfuz,Кайф ты поймала,Konfuz,Кайф ты поймала,[],"Твои подружки хотят ко мне в панамеру\nГоворят мне о любви

In [8]:
df['extracted_brands'] = df['extracted_brands'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# 2. Explode the lists so each dictionary gets its own row.
# Empty lists [] will become NaN, which we drop.
exploded_brands = df['extracted_brands'].explode().dropna()

# 3. Extract the 'normalized' key from each dictionary and get the unique values
distinct_brands = exploded_brands.str.get('normalized').dropna().unique().tolist()

print(distinct_brands)

['BMW', 'Porsche', 'Macallan', 'Nirvana', 'Asphalt 8: Airborne', 'Hermès Birkin', 'Cadillac', 'Uber', 'Sapsan', 'Tinder', 'Instagram', 'Johnnie Walker', 'Badoo', 'Mercedes-Benz', 'Glock', 'Louis Vuitton', 'Zorski', 'Rita Ora', 'LADA', 'Stone Island', 'Honda', 'Rolls-Royce', 'Land Rover', 'TT (Audi TT or TT pistol, but in context likely refers to the car)', 'Nagant (Nagant M1895 revolver)', 'Budweiser', 'Taobao', 'Apple', 'Campari', 'Audi', 'Vans', "M&M's", 'Toyota', 'Loro Piana', 'Timeless', 'Tom Ford', 'TIGO', 'Martini', 'WhatsApp', 'Bunny Boots', 'MASTERMIND', 'PayPal', 'Tommy Hilfiger', 'Calvin Klein', 'New Rock', 'Rick Owens', 'Brabus', 'Parliament Aqua', 'Kylie Jenner', 'Emirates (airline)', 'Durex', 'Bottega Veneta', 'RolePlay', 'Majestic RP', 'Bistro', 'Polka', 'GAZ', 'Bosch', 'Bork', 'Whip', 'Majestic', 'ITAR-TASS', 'Rolex', 'Gucci', 'Avito', 'Google', 'Nike', 'Adidas', 'Cartier', 'Hennessy', 'Louis C.K.', 'Michelin', 'Whisky', 'Prada', 'Moët & Chandon', 'Ferrari', 'Chanel', 'N

In [3]:
import ast

brand_unification_map = {
    # Apple
    'Apple/iPhone': 'Apple', 'Apple/MacBook': 'Apple', 'Apple/Siri': 'Apple', 
    'Apple/FaceTime': 'Apple', 'Apple/iTunes': 'Apple', 'Apple App Store': 'Apple', 
    'Apple AirPods': 'Apple', 'Apple/iBook': 'Apple', 'Apple/iPod': 'Apple', 
    'Apple/Logic Pro': 'Apple', 'Apple/iPad': 'Apple',
    
    # Mercedes
    'Mercedes-Benz G-Class': 'Mercedes-Benz', 'Mercedes-AMG': 'Mercedes-Benz', 
    'Mercedes-Benz S-Class': 'Mercedes-Benz', 'Mercedes-Benz S63 AMG': 'Mercedes-Benz', 
    'Mercedes-Benz SLS AMG': 'Mercedes-Benz', 'Mercedes-Benz GLS': 'Mercedes-Benz', 
    'Mercedes-AMG GT': 'Mercedes-Benz',
    
    # BMW
    'BMW X5': 'BMW', 'BMW X6': 'BMW', 'BMW M5': 'BMW', 'BMW i8': 'BMW', 
    'BMW M2 Competition': 'BMW', 'BMW M': 'BMW',
    
    # Porsche
    'Porsche Macan': 'Porsche', 'Porsche Cayenne': 'Porsche', 'Porsche Panamera': 'Porsche',
    
    # Toyota
    'Toyota Camry': 'Toyota', 'Toyota Land Cruiser': 'Toyota', 
    'Toyota Supra': 'Toyota', 'Toyota Celica': 'Toyota',
    
    # LADA
    'LADA Priora': 'LADA', 'LADA/Zhiguli': 'LADA', 'LADA Niva': 'LADA', 
    'LADA (Жигули)': 'LADA', 'LADA Kalina': 'LADA', 'LADA Zhiguli': 'LADA', 'LADA Largus': 'LADA',
    
    # Chevrolet
    'Chevrolet Corvette': 'Chevrolet', 'Chevrolet Tahoe': 'Chevrolet', 'Chevrolet Camaro': 'Chevrolet',
    
    # Other Cars
    'Lamborghini Urus': 'Lamborghini',
    'Bugatti La Voiture Noire': 'Bugatti',
    'Rolls-Royce Phantom': 'Rolls-Royce', 'Rolls-Royce Cullinan': 'Rolls-Royce',
    'Tesla Roadster': 'Tesla', 'Tesla Cybertruck': 'Tesla',
    'Audi Q7': 'Audi', 
    'Land Rover Range Rover Sport': 'Land Rover',
    'Cadillac Escalade': 'Cadillac',
    'Bentley Bentayga': 'Bentley',
    'Nissan Skyline': 'Nissan',
    'GAZelle': 'GAZ', 'GAZ (ГАЗель)': 'GAZ', 'GAZ Volga': 'GAZ',
    
    # Misc/Food/Drinks/Other
    'Johnnie Walker Black Label': 'Johnnie Walker',
    'Kalashnikov (AK)': 'Kalashnikov',
    'ЦУМ (Central Department Store)': 'TSUM',
    'Haribo Gummy Bears': 'Haribo',
    'Kentucky Fried Chicken (KFC)': 'KFC',
    'Moët & Chandon (Dom Pérignon)': 'Moët & Chandon',
    'Louis Roederer Cristal': 'Cristal',
    'Daytona (Rolex Daytona)': 'Rolex', 'Daytona': 'Rolex'
}

df_genres['extracted_brands'] = df_genres['extracted_brands'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_genres['extracted_brands'] = df_genres['extracted_brands'].apply(
    lambda lst: [
        {**item, 'normalized': brand_unification_map.get(item.get('normalized'), item.get('normalized'))}
        for item in lst
    ] if isinstance(lst, list) else lst
)

In [4]:
df_genres['extracted_brands'] = df_genres['extracted_brands'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

brand_counts = (
    df_genres['extracted_brands']
    .explode()                
    .dropna()                 
    .str.get('normalized')    
    .value_counts()       
)

print(brand_counts)

extracted_brands
Mercedes-Benz                                                                                      70
Apple                                                                                              42
Gucci                                                                                              37
BMW                                                                                                36
Porsche                                                                                            35
Louis Vuitton                                                                                      24
Lamborghini                                                                                        24
TikTok                                                                                             24
Instagram                                                                                          21
Prada                                                            

In [5]:
import pandas as pd

categories = {
    'cars': [
        'BMW', 'Porsche', 'Cadillac', 'LADA', 'Honda', 'Rolls-Royce', 'Land Rover', 'Audi', 
        'Toyota', 'Mercedes-Benz', 'Brabus', 'GAZ', 'Lamborghini', 'Bugatti', 'Tesla', 
        'Bentley', 'Nissan', 'Jeep', 'Ducati', 'Harley-Davidson', 'Maserati', 'Infiniti', 
        'ZIL', 'Renault', 'Foreign Car Brand', 'Fiat', 'Suzuki', 'Jaguar', 'Chevrolet', 
        'McLaren', 'Mini Cooper', 'Ford', 'Dodge', 'Dodge Viper', 'Ferrari'
    ],
    
    'electronics': [
        'Apple', 'Google', 'Bosch', 'Bork', 'Edison', 'Nokia', 'ChatGPT', 'Suno AI', 
        'Bluetooth', 'Laptop/Notebook', 'Siemens', 'Nintendo', 'Bitcoin', 'Adobe Photoshop', 
        'Motorola', 'LG', 'Leica', 'Sony Walkman', 'Microsoft Windows', 
        'Thin Film Transistor (display technology)', 'Dyson'
    ],
    
    'social media': [
        'Uber', 'Tinder', 'Instagram', 'Badoo', 'WhatsApp', 'TikTok', 'Facebook', 'MySpace', 
        'YouTube', 'Twitter', 'Telegram', 'VK (VKontakte)', 'Yandex', 'Spotify', 
        'Instagram Stories', 'TON (Telegram Open Network)', 'Likee', 'Wildberries', 'Ozon', 
        'Taobao', 'Avito', 'Pornhub'
    ],
    
    'luxury clothes': [
        'Hermès Birkin', 'Louis Vuitton', 'Stone Island', 'Vans', 'Loro Piana', 'Tom Ford', 
        'MASTERMIND', 'Tommy Hilfiger', 'Calvin Klein', 'New Rock', 'Rick Owens', 
        'Bottega Veneta', 'Rolex', 'Gucci', 'Nike', 'Adidas', 'Cartier', 'Prada', 'Chanel', 
        'Trapstar', 'Corteiz', 'Goyard', 'Bosco', 'Under Armour', 'Cartier Tank', 
        'Maison Margiela', 'Balenciaga', 'Hood By Air', 'NewRock', 'Dolce & Gabbana', 
        'Armani', 'TSUM', 'Amiri', 'Fendi', 'Christian Louboutin', 'Air Jordan', 
        'Audemars Piguet', 'Versace', 'Tiffany & Co.', 'Lacoste', 'Dior', 'Vetements', 
        'Reebok', 'Ralph Lauren', 'DC Shoes', 'Montblanc', 'Agent Provocateur', 'Ed Hardy', 
        'Swarovski', 'Martine Rose', 'Vinted', 'Polo Ralph Lauren', 'Patek Philippe', 
        'Miu Miu', 'Hermès', 'Mowalola', 'Ottolinger', 'Graff', 'Juicy Couture', 'Gabbana', 
        'Chopard', "Levi's", 'Givenchy', 'HYPEBEAST', 'Supreme and Louis Vuitton', 'Supreme', 
        'Nike Air Max', 'Yeezy', 'Heron Preston', 'Off-White', 'Mackintosh', 'VVS Diamonds', 
        'Kith', 'Ronnie Fieg', 'Hypebeast', 'Paco Rabanne', 'Jimmy Choo', 'Converse', 
        'Burberry', 'Philipp Plein', 'Bulgari', 'Roberto Cavalli', 'Ray-Ban', 
        'Hubert de Givenchy', 'Zara', 'UGG', 'Armani/Milano', 'Raf Simons', 'Pandora', 
        'Brioni', 'Kilian', 'Baccarat', 'Roger Dubuis'
    ],
    
    'weapons': [
        'Glock', 'TT (Audi TT or TT pistol, but in context likely refers to the car)', 
        'Nagant (Nagant M1895 revolver)', 'Ruger', 'MAC-10', 'Winchester', 'Mauser', 
        'Kalashnikov', 'Yarygin', 'Uzi'
    ],
    
    'alcohol and smoke': [
        'Macallan', 'Johnnie Walker', 'Budweiser', 'Campari', 'Martini', 'Parliament Aqua', 
        'Hennessy', 'Whisky', 'Moët & Chandon', 'Chivas Regal', 'Parliament', 'Dom Pérignon', 
        'Champagne', 'Martini & Rossi', 'Marlboro', 'Champagne/Rosé Wine', 'Corona', 
        'Don (beer brand)', 'The Macallan', 'Philip Morris', 'Bacardi', 'Cristal', 
        'XO Cognac', "Jack Daniel's", 'Jameson Irish Whiskey', 'Baltika', 'Swisher Sweets', 
        'Kent cigarettes', 'Veuve Clicquot', 'Tuborg', 
        'Rosé (likely referring to rosé wine/champagne brands like Moët & Chandon, Veuve Clicquot, etc.)', 
        'Starka', 'Amaretto', 'Guinness', 'Rothmans', 'Kent', 'Abrau-Durso', 'Aperol', 
        'Беломорканал', 'Irish Coffee (Baileys Irish Cream or similar)', 'IQOS'
    ],
    
    'food': [
        "M&M's", 'Sprite', 'KFC', 'Red Bull', "McDonald's", 'Nestlé', 'Dirol', 'Hubba Bubba', 
        'Meller', 'Lit Energy', 'Benihana', 'Food City', 'Coca-Cola', 'Pulpy (juice brand)', 
        'Nutella', "Domino's Pizza", 'Tabasco', 'Fanta', 'Haribo', 'Chupa Chups', 
        'Happy Meal', 'Foie Gras', 'Vanilla', 'Club-Mate', 'Nobu', 'Bistro'
    ],
    
    'places': [
        'Sapsan', 'Courchevel', 'Hollywood', 'Beverly Hills', 'Dubai Mall', 
        'Moscow Ring Road', 'United States', 'Kuznetsky Most (shopping district/brand)', 
        'Emirates (airline)'
    ],
    
    'persons': [
        'Rita Ora', 'Kylie Jenner', 'Louis C.K.', 'Sade', 'Akon', 'Eminem', 'Dr. Dre', 
        'Quentin Tarantino', 'Uma Thurman', 'Paulo Coelho', 'Timati', 'Drake', 'Lil Pump', 
        'Vin Diesel', 'Dmitry Nagiev', 'Aarne', 'Katy Perry', 'Tupac Shakur', 'J Mozi', 
        'Terry', 'Scrooge', 'R. Kelly', 'Machine Gun Kelly', 'Queen', 'Yury Dud', 'Fellini', 
        'Kanye West', 'Elon Musk', 'Millie', 'Jennifer Lopez', 'Al Capone', 'Jeffree Star', 
        'Cardi B', 'Instasamka', 'RAF Camora', 'ANIKV', 'SALUKI', 'MORGENSHTERN', 
        'Dr. Alban', 'Kino', 'Modern Talking', 'ABBA', 'The Beatles', 
        'MacGyver (TV series/character)', 'Pink Floyd', 'Madlib', 'Nirvana', 
        'Takashi Murakami', 'Little Big'
    ],
    
    'other': [
        'Asphalt 8: Airborne', 'Zorski', 'Timeless', 'TIGO', 'Bunny Boots', 'RolePlay', 
        'Majestic RP', 'Polka', 'Whip', 'Majestic', 'ITAR-TASS', 'Michelin', 'Rolling Stone', 
        'MTV', 'Whiskas', 'BIC', 'EKX', 'National Basketball Association', 
        'Counter-Strike: Global Offensive', 'Columbia Pictures', 'Oscar', 'Grammy', 
        'Forbes', 'Karaoke Systems', 'Kaspersky', 'Bernley', 'IKEA', 'Black Star', 
        'Compact Disc', 'NASCAR', 'BDF', 'FACEIT', 'Gazprom', 'Dynamo', 'Contra', 
        'Mortal Kombat', 'Blade 2', 'Cry Me a River', 'Barbie', 'Aeroflot', 'Disney', 
        'Genshin Impact', 'Golden Sound', 'Ozempic', "Assassin's Creed", 'SLAANG', 
        'Loota', 'NASA', 'Sonic', 'Namer', 'Spacer', 'Speed Racer', 'Benzomoney', 'Metro', 
        'Visa', 'Mastercard', 'Shazam', 'Real Madrid', 'Vostok', 'Day Blink', 'Bem Bollow', 
        'Booking Machine', 'GQ Magazine', 'Rick and Morty', 'Blocker', 
        'Phone (generic tech product)', 'Jeans', 'Black Friday', 'Mega', 'LEGO', 'MI6', 
        'CIA', 'Mossad', 'U.S. Department of State', 'Creepen', 'Mota', 'Marvel', 
        'War Thunder', 'Boeing', 'SpaceX', 'Macarena', 'Minecraft', 'Bratz Dolls', 'VIP', 
        'Maxim', 'Old Spice', 'Mirage', 'London Goodbye', 'SuperSlots', 'Super5.TV', 
        'Starship Enterprise', 'Viagra', 'Brazzers', 'PayPal', 'Sberbank', 'Cosmopolitan', 
        'Durex'
    ]
}


brand_to_group = {brand: group for group, brands in categories.items() for brand in brands}

In [ ]:
brand_origins = {
    # (Russian / Soviet)
    'LADA': 'Domestic',
    'GAZ': 'Domestic',
    'ZIL': 'Domestic',
    'Bork': 'Domestic',
    'Telegram': 'Domestic',
    'VK (VKontakte)': 'Domestic',
    'Yandex': 'Domestic',
    'TON (Telegram Open Network)': 'Domestic',
    'Wildberries': 'Domestic',
    'Ozon': 'Domestic',
    'Avito': 'Domestic',
    'Bosco': 'Domestic',
    'TSUM': 'Domestic',
    'Nagant (Nagant M1895 revolver)': 'Domestic',
    'Kalashnikov': 'Domestic',
    'Yarygin': 'Domestic',
    'Baltika': 'Domestic',
    'Don (beer brand)': 'Domestic',
    'Abrau-Durso': 'Domestic',
    'Беломорканал': 'Domestic',
    'Lit Energy': 'Domestic',
    'Food City': 'Domestic',
    'Sapsan': 'Domestic',
    'Moscow Ring Road': 'Domestic',
    'Kuznetsky Most (shopping district/brand)': 'Domestic',
    'Dmitry Nagiev': 'Domestic',
    'Timati': 'Domestic',
    'Aarne': 'Domestic',
    'Terry': 'Domestic', 
    'Scrooge': 'Domestic', 
    'Yury Dud': 'Domestic',
    'Instasamka': 'Domestic',
    'SALUKI': 'Domestic',
    'MORGENSHTERN': 'Domestic',
    'Kino': 'Domestic',
    'Little Big': 'Domestic',
    'ITAR-TASS': 'Domestic',
    'Kaspersky': 'Domestic',
    'Bernley': 'Domestic',
    'Black Star': 'Domestic',
    'Gazprom': 'Domestic',
    'Dynamo': 'Domestic',
    'Aeroflot': 'Domestic',
    'Vostok': 'Domestic',
    'Booking Machine': 'Domestic',
    'Benzomoney': 'Domestic',
    'Sberbank': 'Domestic',
    'Super5.TV': 'Domestic',
    'EKX': 'Domestic',
    'Majestic RP': 'Domestic',
    'London Goodbye': 'Domestic',

    # CIS
    'Starka': 'CIS',
    'ANIKV': 'CIS',
    'SuperSlots': 'CIS'
}

In [ ]:
df_genres['extracted_brands'] = df_genres['extracted_brands'].apply(
    lambda lst: [
        {
            **item, 
            'category': brand_to_group.get(item.get('normalized'), 'other'),
            'origin': brand_origins.get(item.get('normalized'), 'Western/Abroad')
        } 
        for item in lst
    ] if isinstance(lst, list) else []
)

In [10]:
df_genres.to_csv('/Users/vasilii/Documents/python/study/masters/flask/dashboard_data/df_genres.csv', index=False, encoding='utf-8')

In [ ]:
df_genres[df_genres.extracted_brands.astype(str) != '[]'].sort_values(by='year', ascending=False).sample(1)

,year,platform,raw_artist,raw_song,matched_artist,matched_song,ai_suggestions,lyrics,genius_id,lrclib_id,genres_yandex,genres_musicbrainz,genres_itunes,genres_lastfm,chosen_genre,extracted_brands
139,2026-01-22,apple,ЛСП,Шиншиллы,ЛСП,Шиншиллы,[],"Помню, раньше дулся (Хм-м), аж лопалась рубашка\nЧто однокурсницы предпочитают мне папашку (Мне? Как?)\nОн им дарит платья всякие, цветы и суши (О, о)\nЖую в общаге сушки, покуда эти шлюшки (Ах)\nСидят, развесив уши (М-м), в дорогих машинах (Вж-ж)\nОдной хотел стать мужем (И что?) — не разрешила (Шлюха)\nРомантик ей не нужен (Нет), ей нужен мужчина\nНашла себе кормушку, а на душе паршиво\n\nС милым рай и в шалаше, но выбирай того, в Porsche\nОн лепит всё, что пожелаешь (Всё), из папье-маше\nПох, что там на душе, если он богат, как шейх\nИ лепит всё, что пожелаешь, (Но) из папье-маше\n\nПока ты в Монте-Карло (О, да) жевала трюфеля\nЛениво, будто коала листья эвкалипта (У-у-у)\nЯ вкалывал, как папа Карло (Папа Карл я), — и вуаля:\nРядом со мной шикарная сеньорита…\nСидит во всеоружии в заряженной машине (У)\nНо чтобы стал ей мужем (Я) — она не заслужила (Нет)\nНу, максимум — подружим (Кхм), пускай живут шиншиллы\nСкажи мне, почему же так на душе паршиво?\n\nС милым рай и в шалаше, но выбирай того, в Porsche\nОн лепит всё, что пожелаешь (Всё), из папье-маше\nПох, что там на душе, если он богат, как шейх\nИ лепит всё, что пожелаешь, (Но) из папье-маше\nТачки из папье-маше, яхты из папье-маше\nОн лепит всё, что пожелаешь (Всё), из папье-маше\nВиллы из папье-маше, дамы из папье-маше\nИ всё, что только пожелаешь, (Но) из папье-маше\n\nПа-па-падап, па-па-па-па, папье-маше",12754289.0,NaN,['rap'],"['hip hop', 'russian rap']",['Hip-Hop/Rap'],"['Hip-Hop', 'electronic', 'dubstep', 'experimental', 'indie']",rap,"[{'mention': 'Porsche', 'normalized': 'Porsche', 'category': 'cars', 'origin': 'Western/Abroad'}]"
2516,2025-12-13,zvuk,Anna Asti,Плачу на техно,Anna Asti,Плачу на техно,[],"У-у-у, во-а, а-а-а\nХа\nУ-у-у, а-а-а\n\nНочью уснуть не могу никак\nИ пришла я на техно, чтобы найти тебя снова тут\nВансы твои, как год назад\nНо рядом Nike уже не мои топчут знакомый мне грув\n\nТы же был лишь моим, лишь моим\nИ мы вместе мечтали купить что-то в «Кузнецком мосту»\nТы же был лишь моим, лишь моим\nANNA ASTI играет для вас, а я слёзы еле держу\n\nПлачу на техно, я плачу на техно\nТы не со мной, слёзы льются на рейве\nПлачу на техно, я плачу на техно\nТы не со мной, слёзы льются на рейве\nРейве, рейве, рей-рей-рейве\nПлачу на техно, я плачу на техно\nТы не со мной, слёзы льются на рейве\nПлачу на техно, я плачу на техно\nТы не со мной, слёзы льются на рейве\n\nУ-у-у\n\nПлачу навзрыд, слёзы бегут по щекам\nИ со щёк кап-кап мне на новый Стонак\nЯ жила лишь тобой, ты сказал мне «Goodbye»\nИ я горько рыдаю, словно закрыли Outline\n\nТы же был лишь моим, лишь моим\nИ мы вместе мечтали купить что-то в «Кузнецком мосту»\nТы же был лишь моим, лишь моим\nANNA ASTI играет для вас, и я слёз уже не держу\n\nПлачу на техно, я плачу на техно\nТы не со мной, слёзы льются на рейве\nПлачу на техно, я плачу на техно\nТы не со мной, слёзы льются на рейве\n\nПлачу на техно (У-у), я плачу на техно (У-у-у)\nТы не со мной, слёзы льются на рейве (У-у-у, э-э)",12807540.0,NaN,['pop'],[],['Pop'],"['pop', 'russian', 'Ukrainian', 'Ukraine', 'Russian Pop']",pop,"[{'mention': 'Вансы', 'normalized': 'Vans', 'category': 'luxury clothes', 'origin': 'Western/Abroad'}, {'mention': 'Nike', 'normalized': 'Nike', 'category': 'luxury clothes', 'origin': 'Western/Abroad'}, {'mention': '«Кузнецком мосту»', 'normalized': 'Kuznetsky Most (shopping district/brand)', 'category': 'places', 'origin': 'Domestic'}, {'mention': 'ANNA ASTI', 'normalized': 'ANNA ASTI (singer/brand)', 'category': 'other', 'origin': 'Western/Abroad'}, {'mention': 'Стонак', 'normalized': 'Stone Island', 'category': 'luxury clothes', 'origin': 'Western/Abroad'}]"
127,2025-11-26,apple,Nextime,Светлана!,Nextime,Светлана!,[],"«СВЕТЛАНА!» — это энергичный и провокационный клубный 